In [1]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [2]:
MY_GENE = "CYP2C19"        # <-- set this to YOUR assigned gene
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on CYP2C19


In [3]:
# TODO Q1
#   (a) if MY_FILE exists, read it; otherwise fetch_gene(MY_GENE) and save it
#   (b) pass it through numeric() so scores are numbers, not strings
#   (c) add a `label` column using clinvar_label()
#   (d) run and print the three quality checks described above
#
# Everything you need is already defined. This is about eight lines.
   # <- your code
# (a) Load or fetch data
if MY_FILE.exists():
    G = pd.read_csv(MY_FILE, sep="\t")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)

# (b) Convert scores to numeric
G = numeric(G)

# (c) Add ClinVar labels
G["label"] = G["clinvar_sig"].map(clinvar_label)

# (d) Three quality checks

# Check 1: Variant count
print("Total variants:", len(G))

# Check 2: Known anchor variants
print("\nKnown anchor variants:")
anchors_for_gene = {
    rsid: allele for rsid, allele in ANCHORS.items()
    if allele.startswith(MY_GENE)
}
for rsid, allele in anchors_for_gene.items():
    found = G["rsid"].astype(str).eq(rsid).any()
    print(rsid, allele, "FOUND" if found else "MISSING")

# Check 3: Score columns
print("\nNon-missing scores:")
print(G[["cadd_phred", "revel", "gnomad_af"]].notna().sum())


Total variants: 80

Known anchor variants:
rs4244285 CYP2C19*2 FOUND
rs12248560 CYP2C19*17 FOUND

Non-missing scores:
cadd_phred    68
revel         54
gnomad_af     55
dtype: int64


In [4]:
# TODO Q2
#   (a) group G by consequence and count non-null scores per tool
#   (b) print the table, plus the overall percentage scored by each tool
#   (c) print which consequence classes have ZERO REVEL scores
#
# Hint: .groupby("consequence") then .apply(lambda s: s.notna().sum())

TOOLS = ["cadd_phred", "revel", "alphamissense"]

# <- your code
# (a) Count available scores by consequence
coverage = G.groupby("consequence")[TOOLS].count()

# (b) Print coverage table and overall percentages
print("Coverage by consequence class:")
print(coverage)

print("\nOverall coverage (%):")
print((G[TOOLS].notna().mean() * 100).round(2))

# (c) Print classes with zero REVEL scores
print("\nClasses with ZERO REVEL scores:")
print(coverage.index[coverage["revel"] == 0].tolist())


Coverage by consequence class:
                       cadd_phred  revel  alphamissense
consequence                                            
intergenic_region               1      0              0
intron_variant                  1      0              0
missense_variant               53     53             54
splice_donor_variant            0      0              0
start_lost                      1      1              0
stop_gained                     1      0              0
stop_lost                       1      0              0
synonymous_variant              6      0              0
upstream_gene_variant           4      0              0

Overall coverage (%):
cadd_phred       85.0
revel            67.5
alphamissense    67.5
dtype: float64

Classes with ZERO REVEL scores:
['intergenic_region', 'intron_variant', 'splice_donor_variant', 'stop_gained', 'stop_lost', 'synonymous_variant', 'upstream_gene_variant']


In [5]:
# TODO Q3
#   (a) set MAX_AF to a threshold you can defend for YOUR gene
#   (b) report how many variants it removes
#   (c) CRITICAL: list the P/LP or "drug response" variants your filter removes.
#       If it removes a known functional allele, either change the threshold or
#       justify keeping it in your report.
#   (d) for contrast, report what a 0.001 rare-disease filter would have removed
#   (e) set G to the filtered set

MAX_AF = 0.25     # <- your choice

# <- your code
# Keep an unfiltered copy for comparison
G_original = G.copy()

# (b) Find variants removed by the chosen threshold
removed = G_original[
    G_original["gnomad_af"] > MAX_AF
]

print("Chosen MAX_AF:", MAX_AF)
print("Total variants:", len(G_original))
print("Removed variants:", len(removed))

# (c) Inspect clinically relevant removed variants
print("\nClinically relevant removed variants:")
important = removed[
    removed["clinvar_sig"].fillna("").str.contains(
        "pathogenic|drug response",
        case=False,
        regex=True
    )
]

print(important[
    ["rsid", "gene", "clinvar_sig", "gnomad_af"]
].to_string(index=False))

# Check known CYP2C19 anchor alleles
print("\nKnown anchors removed:")
print(
    removed[
        removed["rsid"].isin(["rs4244285", "rs12248560"])
    ][["rsid", "gnomad_af"]]
)

# (d) Compare with a 0.001 rare-disease filter
rare_removed = G_original[
    G_original["gnomad_af"] > 0.001
]

print("\nRemoved by 0.001 filter:", len(rare_removed))
print("Important variants removed by 0.001:")
print(
    rare_removed[
        rare_removed["rsid"].isin(
            ["rs4244285", "rs12248560"]
        )
    ][["rsid", "gnomad_af"]]
)

# (e) Apply your chosen filter
G = G_original[
    G_original["gnomad_af"].isna() |
    (G_original["gnomad_af"] <= MAX_AF)
].copy()

print("\nRemaining variants:", len(G))


Chosen MAX_AF: 0.25
Total variants: 80
Removed variants: 2

Clinically relevant removed variants:
Empty DataFrame
Columns: [rsid, gene, clinvar_sig, gnomad_af]
Index: []

Known anchors removed:
Empty DataFrame
Columns: [rsid, gnomad_af]
Index: []

Removed by 0.001 filter: 19
Important variants removed by 0.001:
          rsid  gnomad_af
34   rs4244285   0.167953
63  rs12248560   0.200120

Remaining variants: 78


In [6]:
# TODO Q4
#   (a) choose SCORE_COL and BANDS and run calibrate() on G
#   (b) print how many labelled variants the calibration rests on
#   (c) print how many bands rest on fewer than 10 variants
#   (d) if your gene is too small to calibrate, say so in the output and take
#       one of the two documented routes above
#
# Suggested bands: CADD [0,10,20,25,30,50]   REVEL [0,.2,.4,.6,.8,1.0]

SCORE_COL = "cadd_phred"   # <- your choice
BANDS     = [0, 10, 20, 25, 30, 50]   # <- your bin edges

# <- your code
# (a) Calibrate CADD scores
calibration = calibrate(G, SCORE_COL, BANDS)

print("Calibration results:")
print(calibration)

# (b) Count labelled variants
n_p = calibration.attrs["n_p"]
n_b = calibration.attrs["n_b"]

print("\nPathogenic/Likely pathogenic:", n_p)
print("Benign/Likely benign:", n_b)
print("Total labelled variants:", n_p + n_b)

# (c) Identify bands with fewer than 10 variants
small_bands = calibration[calibration["n"] < 10]

print("\nBands with fewer than 10 variants:")
print(small_bands)

print("\nNumber of small bands:", len(small_bands))

# (d) Check if calibration has sufficient data
if n_p == 0 or n_b == 0:
    print("\nCalibration is not reliable:")
    print("One reference class is missing.")
    print("Consider pooling a second gene or disclosing this limitation.")
elif n_p + n_b < 10:
    print("\nVery small calibration dataset.")
    print("Report this limitation or consider pooling a second gene.")
else:
    print("\nCalibration completed.")
    print("Interpret LR values together with band sample sizes.")

# (d) Small-gene route
print("Small-gene route selected.")
print("Calibration is not reliable: 0 P/LP and 5 B/LB variants.")
print("This limitation will be disclosed in the report.")


Calibration results:
          n  n_PLP  n_BLB  LR+        evidence
band                                          
(0, 10]   5      0      5  0.0  strong, benign
(10, 20]  0      0      0  NaN       undefined
(20, 25]  0      0      0  NaN       undefined
(25, 30]  0      0      0  NaN       undefined
(30, 50]  0      0      0  NaN       undefined

Pathogenic/Likely pathogenic: 0
Benign/Likely benign: 5
Total labelled variants: 5

Bands with fewer than 10 variants:
          n  n_PLP  n_BLB  LR+        evidence
band                                          
(0, 10]   5      0      5  0.0  strong, benign
(10, 20]  0      0      0  NaN       undefined
(20, 25]  0      0      0  NaN       undefined
(25, 30]  0      0      0  NaN       undefined
(30, 50]  0      0      0  NaN       undefined

Number of small bands: 5

Calibration is not reliable:
One reference class is missing.
Consider pooling a second gene or disclosing this limitation.
Small-gene route selected.
Calibration is not relia

In [7]:
# TODO Q5
#   (a) take the VUS rows from G
#   (b) map each one to the evidence strength of its calibrated band
#   (c) count how many move towards pathogenic, towards benign, and stay put
#   (d) export to pgx_classified_<MY_GENE>.csv with these columns:
#       rsid, gene, consequence, aa_change, your score column, gnomad_af,
#       evidence, provisional_call

# <- your code
# (a) Select VUS variants
VUS = G[G["label"] == "VUS"].copy()

print("Total VUS variants:", len(VUS))

# (b) Assign evidence from calibration, if valid
n_p = calibration.attrs["n_p"]
n_b = calibration.attrs["n_b"]

CALIBRATION_VALID = (n_p > 0) and (n_b > 0)

VUS["evidence"] = "unavailable: calibration insufficient"
VUS["provisional_call"] = "VUS"

if CALIBRATION_VALID:
    VUS["band"] = pd.cut(VUS[SCORE_COL], BANDS)
    VUS["evidence"] = VUS["band"].map(
        calibration["evidence"].to_dict()
    ).astype(object).fillna("undefined")

    VUS["provisional_call"] = VUS["evidence"].apply(
        lambda x: "towards pathogenic"
        if "pathogenic" in x
        else ("towards benign" if "benign" in x
              else "VUS")
    )
else:
    print("Small-gene route: no reliable calibrated evidence.")
    print("All VUS remain unclassified.")

# (c) Count classification outcomes
print("\nClassification counts:")
print(VUS["provisional_call"].value_counts())

# (d) Export CSV
export_cols = [
    "rsid", "gene", "consequence", "aa_change",
    SCORE_COL, "gnomad_af", "evidence",
    "provisional_call"
]

OUTPUT_FILE = f"pgx_classified_{MY_GENE}.csv"

VUS[export_cols].to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)
print("Exported variants:", len(VUS))


Total VUS variants: 33
Small-gene route: no reliable calibrated evidence.
All VUS remain unclassified.

Classification counts:
provisional_call
VUS    33
Name: count, dtype: int64

Saved: pgx_classified_CYP2C19.csv
Exported variants: 33


In [8]:
SKIP_ATLAS   = False
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({
                    "hgvs_g": r.hgvs_g,
                    "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                    "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]
                })
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="hgvs_g", how="left", validate="many_to_one")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")

  50/78 queried

scored 70 variants (0 failures)

highest-impact CYP2C19 variants by AVI

       rsid          consequence  avi_phred         top_feature  revel  cadd_phred label
  rs4986893          stop_gained      32.43 PROTEIN_TERMINATION    NaN      35.000 other
rs371241592     missense_variant      25.99       ALPHAMISSENSE  0.518      23.600   VUS
 rs72558186 splice_donor_variant      25.51     MERGED_SPLICING    NaN         NaN other
 rs55640102            stop_lost      25.48           STOP_LOST    NaN      13.030 other
 rs41291556     missense_variant      25.07       ALPHAMISSENSE  0.440      25.700 other
rs760289518     missense_variant      24.22       ALPHAMISSENSE  0.506      25.900   VUS
 rs28399504           start_lost      23.84          START_LOST  0.314      15.940 other
rs371309036     missense_variant      23.75       ALPHAMISSENSE  0.407      24.000   VUS
rs372875966     missense_variant      22.93       ALPHAMISSENSE  0.528      23.000   VUS
  rs4244285   synony

In [9]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

discordant variants in CYP2C19

flag_cadd_vs_revel      3
flag_revel_vs_am        6
flag_scored_but_vus     0
flag_common_but_plp     4
flag_no_tool_at_all    10
flag_atlas_only         4

23 variants trip at least one rule

       rsid           consequence                         consequence_all aa_change  cadd_phred  revel  alphamissense   avi  gnomad_af label
        NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN   NaN        NaN   VUS
  rs7902257 upstream_gene_variant intergenic_region;upstream_gene_variant       NaN         NaN    NaN            NaN  0.18   0.022133 other
        NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN   NaN        NaN  B/LB
 rs41291556      missense_variant                        missense_variant     W120R      25.700  0.440         0.9402 25.07   0.001571 other
rs145328984      missense_variant                        missense_vari